In [ ]:
# =============================================================================
# FedGT-Proto: Federated Graph-Temporal Prototypical Learning for
#              Subject-Disjoint Few-Shot WiFi CSI Human Activity Recognition
#
#             =============================================================
#             = Authors : Yahya Kord Tamandani, Hassan Rezaei             =
#             = Affiliation: University of Sistan and Baluchestan         =
#             = Contact : Yahya.kord@gmail.com | Yahya.kord@cs.usb.ac.ir  =
#             =============================================================
#
# Methods : FedAvg | FedProx | FedProto | FedGT-Proto
#
# HOW TO USE
# ----------
# 1. Download the Alsaify (or compatible) CSI CSV dataset.
# 2. On Google Drive create:
#       MyDrive/FedGTProto_WiFiHAR/raw_dataset/
# 3. Upload CSI CSVs there (e.g. E1_S01_C1_A1_T1.csv; recursive scan is used).
# 4. Open this notebook in Google Colab and run all cells top to bottom.
# 5. If no CSVs are found, synthetic CSI is generated automatically
#    (pipeline test only — not for paper numbers).
#
# Experiment config (matches paper):
#   100 federated rounds, seeds [1, 7, 21, 42, 84],
#   subject-disjoint split, 5-way 5-shot evaluation.
# =============================================================================
#@title 0) Install & imports
import os, sys, re, json, glob, time, copy, math, random, warnings, hashlib
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.manifold import TSNE
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F

matplotlib.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 13, "axes.labelsize": 12,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "pdf.fonttype": 42,
})
warnings.filterwarnings("ignore")

#@title 1) Mount Drive & paths
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive/FedGTProto_WiFiHAR"
for sub in ["raw_dataset", "feature_cache", "checkpoints", "results", "figures"]:
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device = {DEVICE}")
print(f"[INFO] Project root = {DRIVE_ROOT}")
print("[INFO] Place CSI CSVs under:  MyDrive/FedGTProto_WiFiHAR/raw_dataset/")

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

#@title 2) Configuration (paper-scale)
CFG = {
    "seeds": [1, 7, 21, 42, 84],
    "n_rx": 3, "n_subcarriers": 30, "n_packets": 320,
    "n_activities": 12,
    "environments": ["E1", "E2", "E3"],
    "trials_per_activity": 20,
    "graph_seq_len": 128,
    "n_way": 5, "k_shot": 5, "q_query": 10,
    "rounds": 100,
    "clients_per_round": 6,
    "local_episodes": 30,
    "client_lr": 1e-4, "server_lr": 1e-3,
    "clip_norm": 1.0, "fedprox_mu": 0.01,
    "baseline_hidden": (512, 256, 128), "baseline_embed_dim": 256,
    "gt_hidden_width": 64, "gt_low_rank": 6,
    "gt_transformer_layers": 2, "gt_transformer_heads": 4, "gt_embed_dim": 256,
    "gt_client_lr": 3e-4, "gt_server_lr": 3e-3,
    "gt_local_episodes": 40, "gt_rounds": 100,
    "w_supcon": 0.05, "w_proto_align": 0.05, "w_prox": 1e-5,
    "proto_ema": 0.9, "cosine_temperature": 0.07,
    "eval_every": 5, "n_eval_episodes": 200,
}

#@title 3) Data loading (real CSVs or synthetic fallback)
FILENAME_RE = re.compile(
    r"E(?P<env>\d+)_S(?P<subj>\d+)_C(?P<cfg>\d+)_A(?P<act>\d+)_T(?P<trial>\d+)\.csv$", re.I)

def _synth_trial(T, n_rx, n_sub, act, env, subj, seed):
    rng = np.random.default_rng(seed)
    t = np.arange(T)[:, None]
    freq = 0.025 + 0.012 * (act % 6)
    amp = 1.1 + 0.4 * (act % 4)
    offset = (subj * 0.41) % (2 * np.pi)
    noise = 0.50 if env == "E3" else 0.25
    rows = []
    for rx in range(n_rx):
        mag = np.abs(9.5 + amp * np.sin(2 * np.pi * freq * t + rx * 0.7 + offset)
                     + rng.normal(0, noise, (T, n_sub))) + 0.8
        phase = np.cumsum(rng.normal(0, 0.11 + 0.07 * (env == "E3"), (T, n_sub)), 0) + offset
        phase = (phase + np.pi) % (2 * np.pi) - np.pi
        rows.append(mag * np.exp(1j * phase))
    return np.concatenate(rows, axis=1).astype(np.complex64)

def build_or_load_index(cfg):
    raw_dir = os.path.join(DRIVE_ROOT, "raw_dataset")
    rows = []
    for p in glob.glob(os.path.join(raw_dir, "**", "*.csv"), recursive=True):
        m = FILENAME_RE.search(os.path.basename(p))
        if m:
            rows.append({
                "path": p,
                "environment": f"E{int(m.group('env'))}",
                "subject": int(m.group("subj")),
                "activity": int(m.group("act")),
                "trial": int(m.group("trial")),
            })
    if rows:
        print(f"[DATA] Using {len(rows)} REAL CSI files")
        return pd.DataFrame(rows), False
    print("[DATA] No CSVs found — generating synthetic CSI (for pipeline test only)")
    synth_dir = os.path.join(DRIVE_ROOT, "synthetic_dataset")
    os.makedirs(synth_dir, exist_ok=True)
    idx_path = os.path.join(synth_dir, "index.csv")
    if os.path.exists(idx_path):
        return pd.read_csv(idx_path), True
    rows, ctr = [], 0
    n_subj = 30
    for s in range(1, n_subj + 1):
        env = cfg["environments"][(s - 1) % 3]
        for a in range(1, cfg["n_activities"] + 1):
            for t in range(1, cfg["trials_per_activity"] + 1):
                ctr += 1
                C = _synth_trial(cfg["n_packets"], cfg["n_rx"], cfg["n_subcarriers"], a, env, s, ctr)
                fn = f"{env}_S{s}_C1_A{a}_T{t}.npy"
                np.save(os.path.join(synth_dir, fn), C)
                rows.append({"path": os.path.join(synth_dir, fn), "environment": env,
                             "subject": s, "activity": a, "trial": t})
    df = pd.DataFrame(rows)
    df.to_csv(idx_path, index=False)
    return df, True

DATA_INDEX, USING_SYNTHETIC = build_or_load_index(CFG)

def load_C(path, cfg):
    if path.endswith(".npy"):
        return np.load(path)
    df = pd.read_csv(path)
    cols = [f"csi_1_{rx}_{sc}" for rx in range(1, cfg["n_rx"] + 1)
            for sc in range(1, cfg["n_subcarriers"] + 1)]
    # flexible column fallback
    if not all(c in df.columns for c in cols):
        num = df.select_dtypes(include=[np.number])
        if num.shape[1] >= cfg["n_rx"] * cfg["n_subcarriers"]:
            arr = num.values[:, : cfg["n_rx"] * cfg["n_subcarriers"]]
            return arr.astype(np.float32) + 1j * 0.0
        raise ValueError(f"Cannot parse CSI columns in {path}")
    mat = df[cols].astype(str).values
    flat = mat.reshape(-1)
    out = np.empty(len(flat), dtype=complex)
    for i, s in enumerate(flat):
        s = (str(s).strip().replace(" ", "").replace("+-", "-").replace("-+", "-")
             .replace("i", "j").replace("I", "j"))
        try:
            out[i] = complex(s)
        except Exception:
            out[i] = np.nan
    C = out.reshape(mat.shape)
    for m in range(C.shape[1]):
        col = C[:, m]
        mask = ~np.isnan(col.real)
        if mask.any() and not mask.all():
            col[~mask] = col[mask].mean()
            C[:, m] = col
    mag = np.maximum(np.abs(C), 1e-9)
    return mag * np.exp(1j * np.angle(C))

def circular_mean_std(theta, axis=0):
    s, c = np.sin(theta).mean(axis), np.cos(theta).mean(axis)
    mean = np.arctan2(s, c)
    R = np.sqrt(s ** 2 + c ** 2)
    std = np.sqrt(-2 * np.log(np.clip(R, 1e-8, 1.0)))
    return mean, std

def extract_stat(C):
    mag, phase = np.abs(C), np.angle(C)
    feats = []
    for m in range(C.shape[1]):
        feats += [mag[:, m].mean(), mag[:, m].std()]
        mu, sg = circular_mean_std(phase[:, m])
        feats += [float(mu), float(sg)]
    return np.asarray(feats, dtype=np.float32)

def extract_graph(C, seq_len, n_rx, n_sub):
    T = C.shape[0]
    mag = np.abs(C) + 1e-9
    logm = np.log(mag)
    phase = np.angle(C)
    dlog = np.diff(logm, axis=0, prepend=logm[:1])
    feat = np.stack([logm, np.sin(phase), np.cos(phase), dlog], -1)
    idx = np.linspace(0, T - 1, seq_len).astype(int)
    return feat[idx].astype(np.float32)

def build_feature_cache(cfg, index, synthetic):
    cache_dir = os.path.join(DRIVE_ROOT, "feature_cache")
    candidates = sorted(glob.glob(os.path.join(cache_dir, "features_*.npz")) +
                        glob.glob(os.path.join(cache_dir, "feat_*.npz")))
    for path in candidates:
        try:
            d = np.load(path, allow_pickle=True)
            keys = set(d.files)
            if not {"X_stat", "X_graph", "y"}.issubset(keys):
                continue
            X_stat, X_graph, y = d["X_stat"], d["X_graph"], d["y"]
            subject = d["subject"] if "subject" in keys else d.get("SUBJECT")
            env = d["env"] if "env" in keys else np.array(["E1"] * len(y))
            print(f"[CACHE] Loaded {os.path.basename(path)}  X_stat{X_stat.shape}")
            return X_stat, X_graph, y, subject, env, synthetic
        except Exception:
            continue
    print("[CACHE] Extracting features …")
    Xs, Xg, ys, subs, envs = [], [], [], [], []
    for _, row in index.iterrows():
        C = load_C(row["path"], cfg)
        Xs.append(extract_stat(C))
        Xg.append(extract_graph(C, cfg["graph_seq_len"], cfg["n_rx"], cfg["n_subcarriers"]))
        ys.append(row["activity"] - 1)
        subs.append(row["subject"])
        envs.append(row["environment"])
    X_stat, X_graph = np.stack(Xs), np.stack(Xg)
    y, subject, env = np.asarray(ys), np.asarray(subs), np.asarray(envs)
    key = hashlib.md5(str(X_stat.shape).encode()).hexdigest()[:10]
    out = os.path.join(cache_dir, f"features_{key}.npz")
    np.savez_compressed(out, X_stat=X_stat, X_graph=X_graph, y=y,
                        subject=subject, env=env, is_synthetic=synthetic)
    print(f"[CACHE] Wrote {out}")
    return X_stat, X_graph, y, subject, env, synthetic

X_STAT, X_GRAPH, Y, SUBJECT, ENVIRONMENT, USING_SYNTHETIC = \
    build_feature_cache(CFG, DATA_INDEX, USING_SYNTHETIC)
N_FEAT = X_STAT.shape[1]
print(f"[DATA] classes={len(np.unique(Y))}  synthetic={USING_SYNTHETIC}")

#@title 4) Subject-disjoint split & stores
def build_split(subjects, envs, seed=0, hold=0.25):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"s": subjects, "e": envs}).drop_duplicates()
    train, holdout = [], []
    for e, g in df.groupby("e"):
        ss = g["s"].values.copy()
        rng.shuffle(ss)
        k = max(1, int(round(len(ss) * hold)))
        holdout += ss[:k].tolist()
        train += ss[k:].tolist()
    return sorted(set(train)), sorted(set(holdout))

TRAIN_IDS, HOLD_IDS = build_split(SUBJECT, ENVIRONMENT, seed=0)
print(f"[SPLIT] {len(TRAIN_IDS)} clients, {len(HOLD_IDS)} held-out")

class Store:
    def __init__(self, ids, oversample=True):
        self.data = {}
        for cid in ids:
            m = SUBJECT == cid
            Xstat, Xg, y = X_STAT[m].copy(), X_GRAPH[m].copy(), Y[m].copy()
            if oversample:
                counts = np.bincount(y, minlength=CFG["n_activities"])
                maxc = int(counts.max())
                if maxc > 0:
                    extra_s, extra_g, extra_y = [], [], []
                    rng = np.random.default_rng(int(cid) * 97)
                    for c in range(CFG["n_activities"]):
                        need = maxc - counts[c]
                        if need <= 0: continue
                        idx = np.where(y == c)[0]
                        if len(idx) == 0: continue
                        take = rng.choice(idx, size=need, replace=True)
                        extra_s.append(Xstat[take]); extra_g.append(Xg[take]); extra_y.append(y[take])
                    if extra_s:
                        Xstat = np.concatenate([Xstat] + extra_s)
                        Xg = np.concatenate([Xg] + extra_g)
                        y = np.concatenate([y] + extra_y)
            mu, sg = Xstat.mean(0), Xstat.std(0) + 1e-8
            Xstat = ((Xstat - mu) / sg).astype(np.float32)
            self.data[int(cid)] = {"X_stat": Xstat, "X_graph": Xg.astype(np.float32), "y": y}
    def __getitem__(self, i): return self.data[i]
    def ids(self): return list(self.data.keys())

TRAIN_STORE = Store(TRAIN_IDS, oversample=True)
HOLD_STORE  = Store(HOLD_IDS, oversample=False)

#@title 5) Models, losses, servers
def construct_episode(y, n_way, k_shot, q_query, rng, max_tries=30):
    classes = np.unique(y)
    if len(classes) < 2: return None
    n_way = min(n_way, len(classes))
    freq = np.array([(y == c).sum() for c in classes], dtype=float)
    p = 1.0 / (freq + 1e-6); p /= p.sum()
    for _ in range(max_tries):
        chosen = rng.choice(classes, size=n_way, replace=False, p=p)
        s_idx, s_y, q_idx, q_y = [], [], [], []
        ok = True
        for i, c in enumerate(chosen):
            idx = np.where(y == c)[0]
            if len(idx) < k_shot + q_query: ok = False; break
            take = rng.choice(idx, size=k_shot + q_query, replace=False)
            s_idx.extend(take[:k_shot]); s_y.extend([i] * k_shot)
            q_idx.extend(take[k_shot:]); q_y.extend([i] * q_query)
        if ok:
            return (np.asarray(s_idx), np.asarray(s_y),
                    np.asarray(q_idx), np.asarray(q_y), chosen)
    return None

class BaselineMLP(nn.Module):
    def __init__(self, in_dim, hidden, embed_dim, n_classes=None, dropout=0.2):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.LayerNorm(h), nn.ReLU()]
            d = h
        self.backbone = nn.Sequential(*layers)
        self.embed = nn.Linear(d, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(embed_dim, n_classes) if n_classes is not None else None
    def forward(self, x, return_embed=False):
        h = self.backbone(x)
        z = self.embed(self.drop(h))
        if return_embed or self.head is None: return z
        return self.head(z), z

def build_static_adj(n_rx, n_sub):
    N = n_rx * n_sub
    A = np.eye(N, dtype=np.float32)
    for rx in range(n_rx):
        for sc in range(n_sub):
            i = rx * n_sub + sc
            if sc > 0: A[i, i-1] = A[i-1, i] = 1.0
            if sc < n_sub-1: A[i, i+1] = A[i+1, i] = 1.0
    for sc in range(n_sub):
        for r1 in range(n_rx):
            for r2 in range(r1+1, n_rx):
                i, j = r1*n_sub+sc, r2*n_sub+sc
                A[i, j] = A[j, i] = 1.0
    D = np.diag(1.0 / np.sqrt(A.sum(1) + 1e-8))
    return torch.from_numpy(D @ A @ D)

class GraphTemporalEncoder(nn.Module):
    def __init__(self, n_rx, n_sub, seq_len, width, low_rank, n_layers, n_heads, embed_dim):
        super().__init__()
        self.N = n_rx * n_sub
        self.register_buffer("A_static", build_static_adj(n_rx, n_sub))
        self.E1 = nn.Parameter(torch.randn(self.N, low_rank) * 0.01)
        self.E2 = nn.Parameter(torch.randn(low_rank, self.N) * 0.01)
        self.node_proj = nn.Sequential(nn.Linear(4, width), nn.LayerNorm(width), nn.GELU())
        self.temporal = nn.GRU(width, width, num_layers=2, batch_first=True,
                               bidirectional=True, dropout=0.1)
        self.temp_norm = nn.LayerNorm(2 * width)
        self.attn = nn.MultiheadAttention(2 * width, num_heads=n_heads, dropout=0.1, batch_first=True)
        self.attn_norm = nn.LayerNorm(2 * width)
        self.out = nn.Sequential(nn.Linear(2 * width, embed_dim), nn.LayerNorm(embed_dim))
    def adaptive_adj(self):
        return 0.7 * self.A_static + 0.3 * torch.softmax(self.E1 @ self.E2, dim=-1)
    def forward(self, x):
        h = self.node_proj(x)
        h = torch.einsum("ij,btjc->btic", self.adaptive_adj(), h).mean(2)
        h, _ = self.temporal(h)
        h = self.temp_norm(h)
        a, _ = self.attn(h, h, h)
        h = self.attn_norm(h + a)
        return self.out(h.mean(1))

def euclidean_proto_loss(s_z, s_y, q_z, q_y, n_cls):
    protos = torch.stack([s_z[s_y == c].mean(0) for c in range(n_cls)])
    dist = torch.cdist(q_z, protos, p=2) ** 2
    loss = F.cross_entropy(-dist, q_y)
    acc = (dist.argmin(1) == q_y).float().mean()
    return loss, acc, protos

def cosine_proto_loss(s_z, s_y, q_z, q_y, n_cls, temp=0.07):
    s_z, q_z = F.normalize(s_z, -1), F.normalize(q_z, -1)
    protos = F.normalize(torch.stack([s_z[s_y == c].mean(0) for c in range(n_cls)]), -1)
    logits = (q_z @ protos.t()) / temp
    loss = F.cross_entropy(logits, q_y)
    acc = (logits.argmax(1) == q_y).float().mean()
    return loss, acc, protos

def supervised_contrastive(z, y, temp=0.07):
    z = F.normalize(z, -1)
    sim = z @ z.t() / temp
    mask = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    mask.fill_diagonal_(0)
    if mask.sum() < 1: return z.new_zeros(())
    log_prob = sim - torch.logsumexp(sim, 1, keepdim=True)
    return -(mask * log_prob).sum() / mask.sum()

def proximal(params, ref):
    return sum(((p - r.detach()) ** 2).sum() for p, r in zip(params, ref))

class FedAdamServer:
    def __init__(self, model, lr):
        self.model = model
        self.opt = torch.optim.Adam(model.parameters(), lr=lr)
    def step(self, mean_delta):
        self.opt.zero_grad(set_to_none=True)
        with torch.no_grad():
            for p, d in zip(self.model.parameters(), mean_delta):
                p.grad = (-d).clone()
        self.opt.step()
    def state_dict(self): return self.opt.state_dict()
    def load_state_dict(self, sd): self.opt.load_state_dict(sd)

class FedAvgServer:
    def __init__(self, model): self.model = model
    def step(self, mean_delta):
        with torch.no_grad():
            for p, d in zip(self.model.parameters(), mean_delta):
                p.add_(d)
    def state_dict(self): return {}
    def load_state_dict(self, sd): pass

class ProtoMemory:
    def __init__(self, n_cls, dim, ema=0.9):
        self.ema, self.register = ema, {c: None for c in range(n_cls)}
    def update(self, reports):
        acc = defaultdict(list)
        for r in reports:
            for c, (mu, n) in r.items(): acc[c].append((mu, n))
        for c, items in acc.items():
            total = sum(n for _, n in items)
            if total == 0: continue
            mu = sum(m * n for m, n in items) / total
            self.register[c] = mu.detach() if self.register[c] is None else \
                self.ema * self.register[c] + (1 - self.ema) * mu.detach()
    def get(self, c): return self.register.get(c)

#@title 6) Client updates & evaluation
def client_update_classifier(global_model, client, cfg, rng, method, global_ref=None):
    local = copy.deepcopy(global_model).to(DEVICE)
    local.train()
    opt = torch.optim.Adam(local.parameters(), lr=cfg["client_lr"])
    X = torch.tensor(client["X_stat"], dtype=torch.float32)
    y = torch.tensor(client["y"], dtype=torch.long)
    n, losses, accs = len(y), [], []
    for _ in range(cfg["local_episodes"]):
        idx = rng.choice(n, size=min(48, n), replace=False)
        xb, yb = X[idx].to(DEVICE), y[idx].to(DEVICE)
        logits, _ = local(xb)
        loss = F.cross_entropy(logits, yb)
        if method == "FedProx" and global_ref is not None:
            loss = loss + cfg["fedprox_mu"] * proximal(local.parameters(), global_ref)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(local.parameters(), cfg["clip_norm"])
        opt.step()
        losses.append(loss.item()); accs.append((logits.argmax(1) == yb).float().mean().item())
    return local, float(np.mean(losses)), float(np.mean(accs))

def client_update_proto(global_model, client, cfg, rng, method, global_ref=None):
    local = copy.deepcopy(global_model).to(DEVICE)
    local.train()
    opt = torch.optim.Adam(local.parameters(), lr=cfg["client_lr"])
    X = torch.tensor(client["X_stat"], dtype=torch.float32)
    y, losses, accs = client["y"], [], []
    for _ in range(cfg["local_episodes"]):
        ep = construct_episode(y, cfg["n_way"], cfg["k_shot"], cfg["q_query"], rng)
        if ep is None: continue
        s_idx, s_y, q_idx, q_y, _ = ep
        s_z = local(X[s_idx].to(DEVICE), return_embed=True)
        q_z = local(X[q_idx].to(DEVICE), return_embed=True)
        s_yt = torch.tensor(s_y, device=DEVICE); q_yt = torch.tensor(q_y, device=DEVICE)
        loss, acc, _ = euclidean_proto_loss(s_z, s_yt, q_z, q_yt, int(s_yt.max())+1)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(local.parameters(), cfg["clip_norm"])
        opt.step()
        losses.append(loss.item()); accs.append(acc.item())
    if not losses: return local, 0.0, 0.0
    return local, float(np.mean(losses)), float(np.mean(accs))

def client_update_gt(global_model, client, cfg, rng, global_memory, global_ref, round_idx):
    warmup = max(1, int(0.2 * cfg.get("gt_rounds", cfg["rounds"])))
    align_w = cfg["w_proto_align"] * min(1.0, (round_idx + 1) / warmup)
    lr = cfg.get("gt_client_lr", cfg["client_lr"]) * min(
        1.0, (round_idx + 1) / max(1, int(0.1 * cfg.get("gt_rounds", cfg["rounds"]))))
    local = copy.deepcopy(global_model).to(DEVICE)
    local.train()
    opt = torch.optim.Adam(local.parameters(), lr=lr)
    X = torch.tensor(client["X_graph"], dtype=torch.float32)
    y, losses, accs = client["y"], [], []
    for _ in range(cfg.get("gt_local_episodes", cfg["local_episodes"])):
        ep = construct_episode(y, cfg["n_way"], cfg["k_shot"], cfg["q_query"], rng)
        if ep is None: continue
        s_idx, s_y, q_idx, q_y, chosen = ep
        s_z = local(X[s_idx].to(DEVICE)); q_z = local(X[q_idx].to(DEVICE))
        s_yt = torch.tensor(s_y, device=DEVICE); q_yt = torch.tensor(q_y, device=DEVICE)
        n_cls = int(s_yt.max()) + 1
        proto_loss, acc, local_protos = cosine_proto_loss(
            s_z, s_yt, q_z, q_yt, n_cls, cfg["cosine_temperature"])
        sc = supervised_contrastive(torch.cat([s_z, q_z]), torch.cat([s_yt, q_yt]),
                                    cfg["cosine_temperature"])
        align = s_z.new_zeros(())
        if global_memory is not None:
            for i, c in enumerate(chosen):
                g = global_memory.get(int(c))
                if g is not None:
                    align = align + (1 - F.cosine_similarity(
                        local_protos[i].unsqueeze(0), g.to(DEVICE).unsqueeze(0))).mean()
        prox = proximal(local.parameters(), global_ref) if global_ref is not None else 0.0
        loss = proto_loss + cfg["w_supcon"]*sc + align_w*align + cfg["w_prox"]*prox
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(local.parameters(), cfg["clip_norm"])
        opt.step()
        losses.append(loss.item()); accs.append(acc.item())
    report = {}
    local.eval()
    with torch.no_grad():
        z_all = local(X.to(DEVICE)).cpu()
        for c in np.unique(y):
            mask = y == c
            if mask.any(): report[int(c)] = (z_all[mask].mean(0), int(mask.sum()))
    if not losses: return local, 0.0, 0.0, report
    return local, float(np.mean(losses)), float(np.mean(accs)), report

@torch.no_grad()
def evaluate_fewshot(model, store, cfg, n_episodes, distance="euclidean", feature_key="X_stat"):
    model.eval()
    episode_accs, y_true, y_pred = [], [], []
    per_client = defaultdict(list)
    ids = store.ids()
    ep_per = max(1, n_episodes // max(1, len(ids)))
    for cid in ids:
        client = store[cid]
        X = torch.tensor(client[feature_key], dtype=torch.float32)
        y = client["y"]
        rng = np.random.default_rng(int(cid) * 17 + 42)
        for _ in range(ep_per):
            ep = construct_episode(y, cfg["n_way"], cfg["k_shot"], cfg["q_query"], rng)
            if ep is None: continue
            s_idx, s_y, q_idx, q_y, chosen = ep
            if feature_key == "X_stat":
                s_z = model(X[s_idx].to(DEVICE), return_embed=True)
                q_z = model(X[q_idx].to(DEVICE), return_embed=True)
            else:
                s_z = model(X[s_idx].to(DEVICE)); q_z = model(X[q_idx].to(DEVICE))
            s_yt = torch.tensor(s_y, device=DEVICE)
            n_cls = int(s_yt.max()) + 1
            if distance == "cosine":
                s_z, q_z = F.normalize(s_z, -1), F.normalize(q_z, -1)
                protos = F.normalize(torch.stack([s_z[s_yt == c].mean(0) for c in range(n_cls)]), -1)
                pred = (q_z @ protos.t()).argmax(1).cpu().numpy()
            else:
                protos = torch.stack([s_z[s_yt == c].mean(0) for c in range(n_cls)])
                pred = (torch.cdist(q_z, protos, p=2) ** 2).argmin(1).cpu().numpy()
            acc = float((pred == q_y).mean())
            episode_accs.append(acc); per_client[cid].append(acc)
            y_true.extend(chosen[q_y].tolist()); y_pred.extend(chosen[pred].tolist())
    model.train()
    return {"episode_accs": np.asarray(episode_accs), "y_true": np.asarray(y_true),
            "y_pred": np.asarray(y_pred),
            "per_client": {k: float(np.mean(v)) for k, v in per_client.items()}}

def summarize(res, n_act):
    accs = res["episode_accs"]
    if len(accs) == 0:
        return {"accuracy_mean": 0.0, "macro_f1": 0.0, "worst_client": 0.0,
                "per_client": {}, "cm": np.zeros((n_act, n_act), int).tolist()}
    return {
        "accuracy_mean": float(accs.mean()),
        "macro_f1": float(f1_score(res["y_true"], res["y_pred"], average="macro",
                                   labels=list(range(n_act)), zero_division=0)),
        "worst_client": float(min(res["per_client"].values())) if res["per_client"] else 0.0,
        "per_client": res["per_client"],
        "cm": confusion_matrix(res["y_true"], res["y_pred"], labels=list(range(n_act))).tolist(),
    }

#@title 7) Training loop (4 algorithms, 100 rounds, 5 seeds)
METHODS = ["FedAvg", "FedProx", "FedProto", "FedGT-Proto"]
RESULTS = {m: {} for m in METHODS}

def run_one(method, seed, cfg):
    set_seed(seed)
    run_id = f"{method}_seed{seed}"
    ckpt_path = os.path.join(DRIVE_ROOT, "checkpoints", f"{run_id}.pt")

    if method in ("FedAvg", "FedProx"):
        model = BaselineMLP(N_FEAT, cfg["baseline_hidden"], cfg["baseline_embed_dim"],
                            n_classes=cfg["n_activities"]).to(DEVICE)
        server = FedAvgServer(model)
        feat_key, dist, memory = "X_stat", "euclidean", None
        n_rounds = cfg["rounds"]
    elif method == "FedProto":
        model = BaselineMLP(N_FEAT, cfg["baseline_hidden"], cfg["baseline_embed_dim"],
                            n_classes=None).to(DEVICE)
        server = FedAdamServer(model, cfg["server_lr"])
        feat_key, dist, memory = "X_stat", "euclidean", None
        n_rounds = cfg["rounds"]
    else:
        model = GraphTemporalEncoder(
            cfg["n_rx"], cfg["n_subcarriers"], cfg["graph_seq_len"],
            cfg["gt_hidden_width"], cfg["gt_low_rank"],
            cfg["gt_transformer_layers"], cfg["gt_transformer_heads"],
            cfg["gt_embed_dim"]).to(DEVICE)
        server = FedAdamServer(model, cfg.get("gt_server_lr", cfg["server_lr"]))
        feat_key, dist = "X_graph", "cosine"
        memory = ProtoMemory(cfg["n_activities"], cfg["gt_embed_dim"], cfg["proto_ema"])
        n_rounds = cfg.get("gt_rounds", cfg["rounds"])

    history, start = [], 0
    if os.path.exists(ckpt_path):
        try:
            ck = torch.load(ckpt_path, map_location="cpu")
            model.load_state_dict(ck["model"])
            if hasattr(server, "load_state_dict"):
                server.load_state_dict(ck.get("server", {}))
            history, start = ck["history"], ck["round"] + 1
            print(f"[RESUME] {run_id} from round {start}")
        except Exception as e:
            print(f"[START] {run_id} ({e})")

    client_ids = np.asarray(TRAIN_STORE.ids())
    t0 = time.time()
    for r in range(start, n_rounds):
        rng = np.random.default_rng(seed * 100000 + r)
        chosen = rng.choice(client_ids, size=min(cfg["clients_per_round"], len(client_ids)), replace=False)
        global_ref = [p.detach().clone() for p in model.parameters()]
        deltas, losses, accs, reports = [], [], [], []
        for cid in chosen:
            c_rng = np.random.default_rng(seed * 1000000 + r * 1000 + int(cid))
            data = TRAIN_STORE[cid]
            if method in ("FedAvg", "FedProx"):
                loc, loss, acc = client_update_classifier(
                    model, data, cfg, c_rng, method,
                    global_ref if method == "FedProx" else None)
            elif method == "FedProto":
                loc, loss, acc = client_update_proto(model, data, cfg, c_rng, method)
            else:
                loc, loss, acc, rep = client_update_gt(
                    model, data, cfg, c_rng, memory, global_ref, r)
                reports.append(rep)
            deltas.append([lp.detach() - gp for lp, gp in zip(loc.parameters(), global_ref)])
            losses.append(loss); accs.append(acc)
        mean_delta = [torch.stack([d[i] for d in deltas]).mean(0) for i in range(len(deltas[0]))]
        server.step(mean_delta)
        if memory is not None and reports: memory.update(reports)
        hist = {"round": r, "train_loss": float(np.mean(losses)), "train_acc": float(np.mean(accs))}
        if (r + 1) % cfg["eval_every"] == 0 or r == n_rounds - 1:
            ev = evaluate_fewshot(model, HOLD_STORE, cfg, cfg["n_eval_episodes"], dist, feat_key)
            sm = summarize(ev, cfg["n_activities"])
            hist["eval_acc"], hist["eval_f1"] = sm["accuracy_mean"], sm["macro_f1"]
            print(f"  [{method}] seed={seed} round={r+1:3d}  "
                  f"train={hist['train_acc']:.3f}  eval={sm['accuracy_mean']:.3f}  f1={sm['macro_f1']:.3f}")
        history.append(hist)
        torch.save({"model": model.state_dict(),
                    "server": server.state_dict() if hasattr(server, "state_dict") else {},
                    "history": history, "round": r}, ckpt_path)

    final_ev = evaluate_fewshot(model, HOLD_STORE, cfg, cfg["n_eval_episodes"]*2, dist, feat_key)
    final_sm = summarize(final_ev, cfg["n_activities"])
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[DONE] {run_id}  acc={final_sm['accuracy_mean']:.4f}  "
          f"time={(time.time()-t0)/60:.1f} min  params={n_params}")
    return {"history": history, "final": final_sm, "n_params": n_params}

for method in METHODS:
    for seed in CFG["seeds"]:
        print(f"\n======== {method}  seed={seed} ========")
        RESULTS[method][seed] = run_one(method, seed, CFG)

#@title 8) Summary table
rows = []
for m in METHODS:
    accs = [RESULTS[m][s]["final"]["accuracy_mean"] for s in CFG["seeds"]]
    f1s  = [RESULTS[m][s]["final"]["macro_f1"] for s in CFG["seeds"]]
    worst = [RESULTS[m][s]["final"]["worst_client"] for s in CFG["seeds"]]
    rows.append({
        "Method": m,
        "Acc_%": f"{np.mean(accs)*100:.2f} ± {1.96*np.std(accs)/np.sqrt(len(accs))*100:.2f}",
        "Macro-F1": f"{np.mean(f1s):.3f}",
        "Worst-Client_%": f"{np.mean(worst)*100:.1f}",
        "Params": RESULTS[m][CFG["seeds"][0]]["n_params"],
    })
df = pd.DataFrame(rows)
df.to_csv(os.path.join(DRIVE_ROOT, "results", "summary.csv"), index=False)
print("\n===== FINAL SUMMARY =====")
print(df.to_string(index=False))

#@title 9) Three main figures
FIGDIR = os.path.join(DRIVE_ROOT, "figures")
PALETTE = {"FedAvg": "#c0392b", "FedProx": "#e67e22", "FedProto": "#2980b9", "FedGT-Proto": "#1e8449"}
base_seed = CFG["seeds"][0]

def style(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.25, ls="--")
    ax.set_axisbelow(True)

# Fig A — learning curves
fig, ax = plt.subplots(figsize=(9, 5.5))
for m in METHODS:
    curves = []
    for s in CFG["seeds"]:
        h = RESULTS[m][s]["history"]
        rs = [x["round"]+1 for x in h if "eval_acc" in x]
        ac = [x["eval_acc"] for x in h if "eval_acc" in x]
        if rs: curves.append((rs, ac))
    if not curves: continue
    all_r = sorted(set(r for rs,_ in curves for r in rs))
    mat = np.array([[dict(zip(rs,ac)).get(r, np.nan) for r in all_r] for rs,ac in curves])
    mean, std = np.nanmean(mat,0), np.nanstd(mat,0)
    ax.plot(all_r, mean*100, color=PALETTE[m], lw=2.3, label=m)
    ax.fill_between(all_r, (mean-std)*100, (mean+std)*100, color=PALETTE[m], alpha=0.15)
ax.set_xlabel("Federated Round"); ax.set_ylabel("Few-shot Accuracy (%)")
ax.set_title("Convergence (subject-disjoint evaluation)")
ax.legend(frameon=False); style(ax)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "fig_learning_curves.png"))
fig.savefig(os.path.join(FIGDIR, "fig_learning_curves.pdf"))
plt.show()

# Fig B — final accuracy bars
fig, ax = plt.subplots(figsize=(7.5, 5.2))
means, cis = [], []
for m in METHODS:
    accs = [RESULTS[m][s]["final"]["accuracy_mean"] for s in CFG["seeds"]]
    means.append(np.mean(accs)*100)
    cis.append(1.96*np.std(accs)/max(1,np.sqrt(len(accs)))*100)
bars = ax.bar(METHODS, means, yerr=cis, capsize=5, color=[PALETTE[m] for m in METHODS],
              edgecolor="white", alpha=0.92)
for i,(mn,ci) in enumerate(zip(means,cis)):
    ax.text(i, mn+ci+1.2, f"{mn:.1f}", ha="center", fontweight="bold")
ax.set_ylabel("Few-shot Accuracy (%)")
ax.set_title("Final held-out accuracy (mean ± 95% CI)")
style(ax); fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "fig_final_accuracy.png"))
fig.savefig(os.path.join(FIGDIR, "fig_final_accuracy.pdf"))
plt.show()

# Fig C — per-class F1 (all 4, 2×2)
def per_class_f1(cm):
    cm = np.asarray(cm, float)
    out = []
    for c in range(cm.shape[0]):
        tp, fp, fn = cm[c,c], cm[:,c].sum()-cm[c,c], cm[c,:].sum()-cm[c,c]
        p, r = tp/(tp+fp+1e-12), tp/(tp+fn+1e-12)
        out.append(2*p*r/(p+r+1e-12))
    return np.asarray(out)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
names = [f"A{i+1}" for i in range(CFG["n_activities"])]
for ax, m in zip(axes.flat, METHODS):
    f1s = per_class_f1(RESULTS[m][base_seed]["final"]["cm"])
    ax.bar(range(len(f1s)), f1s, color=PALETTE[m], edgecolor="white", alpha=0.92)
    ax.axhline(f1s.mean(), color="#333", ls="--", lw=1.3, label=f"macro={f1s.mean():.3f}")
    ax.set_xticks(range(len(f1s))); ax.set_xticklabels(names, fontsize=8)
    ax.set_ylim(0, 1.05); ax.set_ylabel("F1"); ax.set_title(m, fontweight="bold", color=PALETTE[m])
    ax.legend(frameon=False, loc="upper right"); style(ax)
fig.suptitle("Per-class F1 (held-out few-shot)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "fig_per_class_f1.png"))
fig.savefig(os.path.join(FIGDIR, "fig_per_class_f1.pdf"))
plt.show()

print(f"\n[DONE] Results → {os.path.join(DRIVE_ROOT, 'results')}")
print(f"       Figures → {FIGDIR}")
if USING_SYNTHETIC:
    print("[NOTE] Synthetic data was used. Upload real CSVs to raw_dataset/ for paper numbers.")